# Parlay — Training Notebook
**The RL Negotiation Arena**

Three-stage pipeline:
1. **Generate** — Gemini self-play episodes (diverse, drift-inclusive)
2. **SFT** — Fine-tune Qwen2.5-7B on top episodes (efficiency ≥ 0.60)
3. **GRPO** — RL fine-tuning with Parlay reward functions

This produces the reward curve shown to hackathon judges.

**Runtime:** ~2.5hr on T4 for 100 episodes + 100 GRPO steps. Full 500-step run: ~8hr on A100.

In [ ]:
# Cell 1: Install all dependencies
!pip install -q fastapi uvicorn websockets pydantic aiosqlite google-generativeai fastmcp numpy python-dotenv httpx
!pip install -q trl peft transformers accelerate bitsandbytes datasets huggingface-hub
!pip install -q matplotlib
print('✓ All dependencies installed')

In [ ]:
# Cell 2: Mount Google Drive and set API keys
import os
from google.colab import drive, userdata

drive.mount('/content/drive')

# Set from Colab Secrets (Secrets tab in left sidebar)
os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
os.environ['HF_TOKEN']       = userdata.get('HF_TOKEN')

# Verify
assert os.environ.get('GOOGLE_API_KEY'), 'GOOGLE_API_KEY not set in Colab Secrets!'
print('✓ API keys loaded')
print(f'  GOOGLE_API_KEY: {os.environ["GOOGLE_API_KEY"][:8]}...')

In [ ]:
# Cell 3: Clone or mount Parlay repo
import os

# Option A: Clone from GitHub (if published)
# !git clone https://github.com/your-username/parlay /content/parlay
# %cd /content/parlay

# Option B: Mount from Google Drive (recommended for hackathon)
PARLAY_PATH = '/content/drive/MyDrive/parlay'  # adjust to your Drive path
if os.path.exists(PARLAY_PATH):
    os.chdir(PARLAY_PATH)
    print(f'✓ Working in {os.getcwd()}')
else:
    # Fallback: work from current directory
    print(f'Working in {os.getcwd()}')
    print('Note: Upload parlay/ to Google Drive at MyDrive/parlay for persistence')

import sys
sys.path.insert(0, os.getcwd())
print('✓ sys.path updated')

In [ ]:
# Cell 4: Verify core imports
from parlay_env import VERSION
from parlay_env.models import PersonaType, TacticalMove
from parlay_env.game_theory import compute_zopa, compute_nash_bargaining_solution
from game.scenarios import SCENARIOS
from agent.personas import PERSONAS

print(f'✓ Parlay v{VERSION} loaded')
print(f'  Personas: {[p.value for p in PersonaType]}')
print(f'  Scenarios: {[s.id for s in SCENARIOS]}')

# Quick game theory sanity check
zopa = compute_zopa(165_000, 125_000)
nash = compute_nash_bargaining_solution(165_000, 125_000)
print(f'\nSaaS ZOPA: {zopa}')
print(f'Nash point: {nash:,.0f}')

In [ ]:
# Cell 5: Initialize database
import asyncio
from scripts.init_db import init_db

await init_db()  # Colab supports top-level await
print('✓ Database initialized')

In [ ]:
# Cell 6: Generate training data (100 episodes for demo)
# For full training, use --episodes 2000
import subprocess
result = subprocess.run(
    ['python', '-m', 'training.generate_data', '--episodes', '100', '--output', 'data/episodes.jsonl'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

In [ ]:
# Cell 7: Inspect generated data
import json
from pathlib import Path

records = []
with open('data/episodes.jsonl') as f:
    for line in f:
        records.append(json.loads(line))

print(f'Total episodes: {len(records)}')
efficiencies = [r['deal_efficiency'] for r in records]
rewards = [r['reward'] for r in records]
above_thresh = sum(1 for e in efficiencies if e >= 0.60)

print(f'Above 0.60 threshold: {above_thresh} ({above_thresh/len(records)*100:.1f}%)')
print(f'Mean efficiency: {sum(efficiencies)/len(efficiencies):.3f}')
print(f'Mean reward: {sum(rewards)/len(rewards):.2f}')
print(f'\nSplit distribution:')
from collections import Counter
print(Counter(r['split'] for r in records))
print('\nPersona distribution:')
print(Counter(r['persona'] for r in records))

In [ ]:
# Cell 8: SFT warmup (Stage 1)
# Requires GPU. Fine-tunes Qwen2.5-7B on top episodes.
import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

!python -m training.sft_train \
    --model Qwen/Qwen2.5-7B-Instruct \
    --data data/episodes.jsonl \
    --output models/parlay-sft \
    --threshold 0.60

In [ ]:
# Cell 9: GRPO fine-tuning (Stage 2 — core RL)
# This produces the reward curve shown to judges.
# Use --steps 100 for demo, --steps 500 for full training.
!python -m training.grpo_train \
    --model models/parlay-sft \
    --data data/episodes.jsonl \
    --output models/parlay-grpo \
    --steps 100

In [ ]:
# Cell 10: Evaluate all three models + plot reward curve
import asyncio
from training.evaluate import compare_models, plot_results
from pathlib import Path
import json

results = await compare_models(
    base='Qwen/Qwen2.5-7B-Instruct',
    sft='models/parlay-sft',
    grpo='models/parlay-grpo',
    n=30,
    data_path='data/episodes.jsonl',
)

# Save results
Path('results').mkdir(exist_ok=True)
with open('results/eval_results.json', 'w') as f:
    json.dump(results, f, indent=2)

# Plot
plot_results(results, Path('results'))

# Display
from IPython.display import Image, display
display(Image('results/reward_curve.png'))

In [ ]:
# Cell 11: Print summary statistics
models = ['Base', 'SFT', 'GRPO']
keys = ['base', 'sft', 'grpo']

print('='*60)
print('PARLAY TRAINING RESULTS SUMMARY')
print('='*60)
for k, m in zip(keys, models):
    r = results[k]
    print(f'\n{m}:')
    print(f'  Mean Reward:     {r["mean_reward"]:>8.2f}')
    print(f'  Mean Efficiency: {r["mean_efficiency"]:>8.3f}')
    print(f'  Deal Close Rate: {r["deal_close_rate"]:>8.1%}')
    print(f'  Above BATNA:     {r["above_batna_rate"]:>8.1%}')

print('\n' + '='*60)
base_r = results['base']['mean_reward']
grpo_r = results['grpo']['mean_reward']
base_e = results['base']['mean_efficiency']
grpo_e = results['grpo']['mean_efficiency']
print(f'Base → GRPO reward improvement:     {grpo_r - base_r:+.1f}')
print(f'Base → GRPO efficiency improvement: {(grpo_e - base_e)*100:+.1f}%')
print('='*60)

In [ ]:
# Cell 12: Push to HuggingFace Hub
import os
HF_REPO = 'your-username/parlay-negotiator'  # <-- Change this!

!python -m training.push_to_hub \
    --model models/parlay-grpo \
    --repo {HF_REPO}

In [ ]:
# Cell 13: Quick demo — play one episode from CLI
!python -m agent.runner --persona shark --scenario saas_enterprise --steps 10

## Next Steps

1. **Play the game**: `uvicorn main:app --port 8000` and open `http://localhost:8000`
2. **MCP integration**: `python -m mcp_server.server stdio` for Claude Desktop
3. **Full training**: Increase `--episodes 2000` and `--steps 500` for production-quality curves
4. **HF Hub**: Share your trained model at `https://huggingface.co/your-username/parlay-negotiator`

---
*Parlay — The arena where AIs learn to close.*